# Data Pipeline Project — Overview

> **Purpose:** This notebook documents the full architecture, workflow, and implementation requirements for our collaborative data pipeline project. It is intended to serve as a living reference document for all team members. Feel free to update this notebook and push it to our Git.
>
> **NOTE:** The code and specific script architecture are merely suggestions, not strict guidelines. Do what is necessary for each step of the pipeline, not just what is given here.

---

## Section 1 — Project Overview

### 1.1 What We Are Building

We are building an **end-to-end, fully online data pipeline** that:

1. Downloads data from a public REST API
2. Cleans and normalizes that data using a set of modular cleaning functions
3. Stores cleaned data as CSV files with meaningful date-range names
4. Loads data into a structured relational database that eliminates redundancy
5. Serves a live web dashboard for visualization and read-only querying
6. Runs on an automated schedule to pull in new data without manual intervention

The entire project is cloud-hosted and collaborative — no teammate needs to run any script locally once the pipeline is set up. We will ONLY be using the GitHub where this file is located for file uploads and updates (except for the final presentation slides, which we can do on Google Slides). 

---

### 1.2 Services and Tools

| Layer | Service | Purpose |
|---|---|---|
| **Version control** | GitHub | Stores all code; triggers deployments and scheduled jobs |
| **Raw + cleaned data storage** | Supabase Storage (buckets) | Holds intermediate CSV files accessible to all scripts |
| **Relational database** | Supabase PostgreSQL | Final normalized database; queried by the dashboard |
| **Scheduled automation** | GitHub Actions | Runs the incremental fetch → clean → port pipeline on a cron schedule |
| **Dashboard hosting** | Streamlit Community Cloud | Auto-deploys the dashboard from GitHub; queries Supabase live |

All services have a **free tier** that is sufficient for this project. Every team member needs only a GitHub account to collaborate. Github, Supabase, and Streamlit Community Cloud are easy to use and understand,

---

### 1.3 Repository Structure

```
project-repo/
│
├── fetch_all.py            # One-time: fetches full historical data in batches
├── fetch_new.py            # Scheduled: fetches only new data since last run
│
├── clean_data.py           # Reads raw CSV → applies cleaning functions → saves cleaned CSV
├── cleaning/
│   ├── __init__.py
│   ├── normalize_caps.py   # One cleaning concern per file
│   ├── strip_whitespace.py
│   ├── parse_dates.py
│   └── ... (one file per cleaning function)
│
├── setup_db.py             # One-time: creates schema in Supabase
├── port_data.py            # Loads cleaned CSV into the database (with dedup)
│
├── streamlit_app.py        # Dashboard: visualizations + read-only queries
│
├── .github/
│   └── workflows/
│       └── update_pipeline.yml   # GitHub Actions cron workflow
│
├── requirements.txt
└── README.md
```

---

### 1.4 Full Pipeline Workflow

The pipeline has two modes: **initial setup** (run once) and **scheduled updates** (run automatically).

**Initial setup sequence (manual, run once):**
```
fetch_all.py  →  clean_data.py  →  setup_db.py  →  port_data.py
```

**Scheduled update sequence (automated via GitHub Actions):**
```
fetch_new.py  →  clean_data.py  →  port_data.py
```

The dashboard (`streamlit_app.py`) runs continuously on Streamlit Community Cloud and always reflects the current state of the database.

---

### 1.5 Data Flow Summary

```
REST API
   │
   ▼
fetch_all.py / fetch_new.py
   │  saves  raw/YYYY-MM-DD_YYYY-MM-DD.csv
   ▼
Supabase Storage  (raw bucket)
   │
   ▼
clean_data.py
   │  saves  cleaned/YYYY-MM-DD_YYYY-MM-DD.csv
   ▼
Supabase Storage  (cleaned bucket)
   │
   ▼
port_data.py  (INSERT ... ON CONFLICT DO NOTHING)
   │
   ▼
Supabase PostgreSQL  (normalized tables)
   │
   ▼
streamlit_app.py  →  Streamlit Community Cloud  →  Users
```

---
## Section 2 — Fetching Raw Data from the API

### 2.1 How the REST API Works

A REST API exposes data over HTTP. You send a `GET` request to a URL (the **endpoint**) with optional **query parameters** that filter or paginate the results. The API responds with data — typically in JSON format — which we then parse and save.

Key concepts relevant to our pipeline:

- **Endpoint:** The base URL for the data resource (e.g., `https://api.example.com/v1/records`)
- **Query parameters:** Filters appended to the URL (e.g., `?since=2024-01-01&limit=1000`)
- **Pagination:** Most APIs do not return all records in one response. They return a page of results plus a cursor or page number to request the next page. We must loop until all pages are retrieved.
- **Rate limiting:** APIs typically cap how many requests you can make per minute. We must respect this with `time.sleep()` calls between requests.
- **Authentication:** Most APIs require an API key passed as a header (e.g., `Authorization: Bearer <key>`) or query parameter. Keys must be stored as GitHub Secrets — never hardcoded in source files.

---

### 2.2 `fetch_all.py` — Full Historical Fetch

**Purpose:** Download the complete history of data from the API. This script is run **once** during initial setup (and again if the database ever needs to be rebuilt from scratch).

**Key requirements:**

- **Batch by date range.** Rather than fetching everything in one giant request, the script divides time into fixed windows (e.g., monthly or quarterly batches) and fetches one window at a time.
- **Resume on failure.** The script tracks progress in a small state file (`fetch_progress.json` in Supabase Storage or locally). If the script crashes at batch 47 of 200, re-running it picks up at batch 47 instead of starting over.
- **Save each batch as a CSV.** Each batch is saved as `raw/YYYY-MM-DD_YYYY-MM-DD.csv` in Supabase Storage, where the dates reflect the *data's date range*, not the date the script ran.
- **Log every batch.** Print (or log to a file) which date range was fetched, how many records were returned, and whether the save succeeded.

**Skeleton structure:**

In [ ]:
# fetch_all.py — skeleton (not yet implemented)

import requests
import pandas as pd
from supabase import create_client
import time, json, os
from datetime import date, timedelta

SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_KEY"]
API_KEY      = os.environ["API_KEY"]
API_BASE_URL = "https://api.example.com/v1/records"
BATCH_DAYS   = 90  # fetch 90-day windows at a time

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

def fetch_batch(start_date: date, end_date: date) -> pd.DataFrame:
    """Fetch all records between start_date and end_date, handling pagination."""
    records = []
    page = 1
    while True:
        response = requests.get(
            API_BASE_URL,
            headers={"Authorization": f"Bearer {API_KEY}"},
            params={"since": str(start_date), "until": str(end_date), "page": page, "limit": 1000}
        )
        response.raise_for_status()
        data = response.json()["results"]
        if not data:
            break
        records.extend(data)
        page += 1
        time.sleep(0.5)  # respect rate limits
    return pd.DataFrame(records)

def save_to_storage(df: pd.DataFrame, start_date: date, end_date: date):
    """Upload a CSV to Supabase Storage with a date-range filename."""
    filename = f"{start_date}_{end_date}.csv"
    csv_bytes = df.to_csv(index=False).encode("utf-8")
    supabase.storage.from_("raw").upload(filename, csv_bytes, {"upsert": "true"})
    print(f"Saved {len(df)} rows → raw/{filename}")

def generate_batches(overall_start: date, overall_end: date):
    """Yield (start, end) tuples for each batch window."""
    current = overall_start
    while current < overall_end:
        batch_end = min(current + timedelta(days=BATCH_DAYS), overall_end)
        yield current, batch_end
        current = batch_end + timedelta(days=1)

# --- Main ---
# TODO: set overall start/end, load progress state, loop through batches
print("fetch_all.py skeleton loaded — implementation pending")

---

### 2.3 `fetch_new.py` — Incremental Fetch

**Purpose:** Fetch only the records that have been added to the API since the last successful run. This script is triggered **automatically** by GitHub Actions on a scheduled interval.

**Key requirements:**

- **Determine the "since" date reliably.** The preferred approach is to query the database for `MAX(record_date)` — the latest date already stored. This is more robust than a timestamp file because it self-corrects if a previous run partially failed.
- **Fall back gracefully.** If the database is empty (e.g., after a schema reset), the script should fall back to a configured default start date rather than crashing.
- **Save output as a date-range CSV.** The filename should reflect the data's date range, e.g., `raw/2024-07-01_2024-07-15.csv`.
- **Exit cleanly if there is nothing new.** If the API returns zero records, log that and exit with code 0 — do not create an empty CSV or fail the GitHub Actions run.

**Why query `MAX(record_date)` instead of saving a timestamp file:**

If a GitHub Actions run fetches data but then `port_data.py` fails before inserting it, the timestamp file would show the fetch as complete while the database has a gap. Querying `MAX(record_date)` from the database directly exposes that gap and the next run will re-fetch it.

**Skeleton structure:**

In [ ]:
# fetch_new.py — skeleton (not yet implemented)

import requests
import pandas as pd
from supabase import create_client
import time, os
from datetime import date, timedelta

SUPABASE_URL    = os.environ["SUPABASE_URL"]
SUPABASE_KEY    = os.environ["SUPABASE_KEY"]
API_KEY         = os.environ["API_KEY"]
API_BASE_URL    = "https://api.example.com/v1/records"
DEFAULT_START   = date(2020, 1, 1)  # fallback if DB is empty

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

def get_last_record_date() -> date:
    """Query the DB for the most recent record date already stored."""
    result = supabase.table("records").select("record_date").order("record_date", desc=True).limit(1).execute()
    if result.data:
        return date.fromisoformat(result.data[0]["record_date"])
    print("Database is empty — falling back to DEFAULT_START")
    return DEFAULT_START

def fetch_since(since: date, until: date) -> pd.DataFrame:
    """Fetch all records between since and until, handling pagination."""
    records = []
    page = 1
    while True:
        response = requests.get(
            API_BASE_URL,
            headers={"Authorization": f"Bearer {API_KEY}"},
            params={"since": str(since), "until": str(until), "page": page, "limit": 1000}
        )
        response.raise_for_status()
        data = response.json()["results"]
        if not data:
            break
        records.extend(data)
        page += 1
        time.sleep(0.5)
    return pd.DataFrame(records)

# --- Main ---
# TODO: get since date, fetch, save CSV, exit cleanly if empty
print("fetch_new.py skeleton loaded — implementation pending")

---
## Section 3 — Cleaning the Data

### 3.1 Design Philosophy

Each cleaning operation is implemented as its own **small, focused function** in its own file inside the `cleaning/` package. This design:

- Makes each function easy to test in isolation
- Makes it easy to turn individual cleaning steps on or off
- Makes the pipeline readable — `clean_data.py` reads like a checklist
- Avoids one giant function that is hard to debug

IF it is preferred for cleaning to be performed in one file (all cleaning helper functions are built directly in `clean_data.py` instead of separately in different files, feel free.

Every cleaning function must follow this contract:

```python
def clean_<something>(df: pd.DataFrame) -> pd.DataFrame:
    """One-line description of what this function does."""
    # ... transformation ...
    return df
```

Functions take a DataFrame and return a DataFrame. They do not modify the input in place.

---

### 3.2 Planned Cleaning Functions

The following cleaning steps will be implemented. The exact column names will be filled in once we have confirmed the API schema.

| File | Function | What It Does |
|---|---|---|
| `normalize_caps.py` | `normalize_caps(df)` | Lowercases (or title-cases) all string columns |
| `strip_whitespace.py` | `strip_whitespace(df)` | Strips leading/trailing whitespace from all string fields |
| `parse_dates.py` | `parse_dates(df)` | Parses date strings into proper `datetime` objects |
| `drop_nulls.py` | `drop_nulls(df)` | Drops rows where required fields are null |
| `normalize_nulls.py` | `normalize_nulls(df)` | Converts empty strings, `"N/A"`, `"null"` etc. to `None` |
| `remove_duplicates.py` | `remove_duplicates(df)` | Drops exact duplicate rows within a single CSV batch |
| `validate_types.py` | `validate_types(df)` | Coerces numeric fields to correct types; flags/drops invalid rows |
| `normalize_ids.py` | `normalize_ids(df)` | Ensures ID fields are consistent types (e.g., all strings, no mixed int/str) |

---

### 3.3 `clean_data.py` — The Orchestrator

This script reads a raw CSV from Supabase Storage, applies all cleaning functions in sequence, and saves the result as a cleaned CSV. It is called with a filename argument so the same script works for any batch.

**Key requirements:**
- Accept the target filename as a command-line argument or environment variable so GitHub Actions can pass it in dynamically.
- Log the row count before and after cleaning so we can detect unexpected data loss.
- Save the cleaned CSV to the `cleaned/` bucket in Supabase Storage with the **same filename** as the raw CSV (just in a different bucket/folder).

**Skeleton structure:**

In [ ]:
# clean_data.py — skeleton (not yet implemented)

import pandas as pd
from supabase import create_client
import os, io

# Import all cleaning functions
from cleaning.normalize_caps    import normalize_caps
from cleaning.strip_whitespace  import strip_whitespace
from cleaning.parse_dates       import parse_dates
from cleaning.drop_nulls        import drop_nulls
from cleaning.normalize_nulls   import normalize_nulls
from cleaning.remove_duplicates import remove_duplicates
from cleaning.validate_types    import validate_types
from cleaning.normalize_ids     import normalize_ids

SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_KEY"]
TARGET_FILE  = os.environ["TARGET_FILE"]  # e.g. "2024-01-01_2024-03-31.csv"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Ordered list of cleaning steps
CLEANING_PIPELINE = [
    normalize_nulls,
    strip_whitespace,
    normalize_caps,
    parse_dates,
    validate_types,
    normalize_ids,
    drop_nulls,
    remove_duplicates,
]

def load_raw_csv(filename: str) -> pd.DataFrame:
    """Download a raw CSV from Supabase Storage and return as a DataFrame."""
    response = supabase.storage.from_("raw").download(filename)
    return pd.read_csv(io.BytesIO(response))

def save_cleaned_csv(df: pd.DataFrame, filename: str):
    """Upload cleaned DataFrame to the 'cleaned' bucket in Supabase Storage."""
    csv_bytes = df.to_csv(index=False).encode("utf-8")
    supabase.storage.from_("cleaned").upload(filename, csv_bytes, {"upsert": "true"})
    print(f"Saved cleaned CSV → cleaned/{filename}")

def run_cleaning_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    """Apply all cleaning functions in order."""
    print(f"Starting cleaning — {len(df)} rows")
    for func in CLEANING_PIPELINE:
        df = func(df)
        print(f"  After {func.__name__}: {len(df)} rows")
    return df

# --- Main ---
# TODO: load raw CSV, run pipeline, save cleaned CSV
print("clean_data.py skeleton loaded — implementation pending")

---
## Section 4 — Database Design

### 4.1 Design Goals

The database schema must satisfy three properties:

1. **No redundancy (normalization).** The same piece of information should be stored in exactly one place. For example, if a record references a city, the city's metadata (population, state, etc.) should be in its own table — not repeated on every record.
2. **Data integrity.** Foreign key constraints, NOT NULL constraints, and unique constraints should be used to prevent invalid data from entering the database.
3. **Queryability.** The schema should make common dashboard queries fast and simple — avoid deeply nested structures that require many joins.

---

### 4.2 Normalization (Plain English)

Normalization means splitting data into tables so that each fact is stored exactly once. The standard levels are:

- **1NF:** Each cell contains one value (no comma-separated lists inside a cell).
- **2NF:** Each non-key column depends on the *whole* primary key, not just part of it.
- **3NF:** No column depends on another non-key column (no transitive dependencies).

For our project, we target **3NF**. In practice this means: if two records share the same category, location, or entity, that shared data lives in a lookup table with a foreign key — not duplicated on every row.

---

### 4.3 Schema Placeholder

> **Note:** The concrete schema will be defined once the API structure is confirmed. The code below shows a *template* for what a normalized schema looks like. Column names and table names will be replaced with real ones.

A typical normalized schema for this type of pipeline looks like:

```sql
-- Lookup / dimension tables (small, stable data)
CREATE TABLE categories (
    id      SERIAL PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);

CREATE TABLE locations (
    id      SERIAL PRIMARY KEY,
    city    TEXT NOT NULL,
    state   TEXT,
    country TEXT NOT NULL
);

-- Main fact table (large, grows over time)
CREATE TABLE records (
    id            TEXT PRIMARY KEY,       -- unique ID from the API
    record_date   DATE NOT NULL,
    value         NUMERIC,
    category_id   INT REFERENCES categories(id),
    location_id   INT REFERENCES locations(id),
    created_at    TIMESTAMPTZ DEFAULT NOW()
);

-- Index for the most common query pattern
CREATE INDEX idx_records_date ON records(record_date);
```

---

### 4.4 `setup_db.py` — Schema Creation

**Purpose:** Connect to Supabase and run the SQL commands that create all tables, indexes, and constraints. This script is run **once** during initial setup.

**Key requirements:**
- Use `CREATE TABLE IF NOT EXISTS` so the script is safely re-runnable without destroying existing data.
- Define all foreign key constraints at creation time.
- Add a `UNIQUE` constraint on the API's native record ID column — this is what enables the deduplication `ON CONFLICT` clause in `port_data.py`.

**Skeleton structure:**

In [ ]:
# setup_db.py — skeleton (not yet implemented)

import os
import psycopg2  # direct PostgreSQL connection for DDL statements

DB_URL = os.environ["DATABASE_URL"]  # Supabase direct connection string

SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS categories (
    id   SERIAL PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS locations (
    id      SERIAL PRIMARY KEY,
    city    TEXT NOT NULL,
    state   TEXT,
    country TEXT NOT NULL,
    UNIQUE(city, country)
);

CREATE TABLE IF NOT EXISTS records (
    id            TEXT PRIMARY KEY,
    record_date   DATE NOT NULL,
    value         NUMERIC,
    category_id   INT REFERENCES categories(id),
    location_id   INT REFERENCES locations(id),
    created_at    TIMESTAMPTZ DEFAULT NOW()
);

CREATE INDEX IF NOT EXISTS idx_records_date ON records(record_date);
"""

def setup_database():
    conn = psycopg2.connect(DB_URL)
    cursor = conn.cursor()
    cursor.execute(SCHEMA_SQL)
    conn.commit()
    cursor.close()
    conn.close()
    print("Schema created successfully.")

# TODO: call setup_database() in main block
print("setup_db.py skeleton loaded — implementation pending")

---
## Section 5 — Porting Data into the Database

### 5.1 Overview

`port_data.py` reads a cleaned CSV from Supabase Storage and inserts its rows into the normalized PostgreSQL tables. It is called both during initial setup (after `fetch_all.py` + `clean_data.py`) and during every scheduled update.

---

### 5.2 Deduplication Strategy

Even though `fetch_new.py` only requests new data, duplicates can still appear in edge cases:

- The API's `since=` filter might use inclusive boundaries, returning one already-seen record
- A GitHub Actions run might be retried after a partial failure
- The same CSV might be manually re-ported during debugging

We guard against all of these with a single SQL clause:

```sql
INSERT INTO records (...) VALUES (...)
ON CONFLICT (id) DO NOTHING;
```

This requires that the `records` table has `id` as a `PRIMARY KEY` or with a `UNIQUE` constraint. Any row whose `id` already exists in the table is silently skipped — no error, no duplicate.

---

### 5.3 Handling Normalized Tables

Because the schema splits data into multiple tables (e.g., `categories`, `locations`, `records`), `port_data.py` must insert in the correct order:

1. Insert into lookup/dimension tables first (`categories`, `locations`)
2. Retrieve the auto-generated IDs for those rows
3. Insert into the main `records` table using those IDs as foreign keys

Use `INSERT ... ON CONFLICT DO NOTHING RETURNING id` to handle the case where a lookup value (e.g., a category name) already exists.

---

### 5.4 `port_data.py` — Skeleton

In [ ]:
# port_data.py — skeleton (not yet implemented)

import pandas as pd
import psycopg2
from supabase import create_client
import os, io

SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_KEY"]
DB_URL       = os.environ["DATABASE_URL"]
TARGET_FILE  = os.environ["TARGET_FILE"]  # e.g. "2024-01-01_2024-03-31.csv"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

def load_cleaned_csv(filename: str) -> pd.DataFrame:
    response = supabase.storage.from_("cleaned").download(filename)
    return pd.read_csv(io.BytesIO(response))

def upsert_lookup(cursor, table: str, column: str, value: str) -> int:
    """Insert a lookup value if it doesn't exist, return its id."""
    cursor.execute(
        f"INSERT INTO {table} ({column}) VALUES (%s) ON CONFLICT ({column}) DO NOTHING RETURNING id",
        (value,)
    )
    result = cursor.fetchone()
    if result:
        return result[0]
    cursor.execute(f"SELECT id FROM {table} WHERE {column} = %s", (value,))
    return cursor.fetchone()[0]

def port_dataframe(df: pd.DataFrame):
    """Insert all rows from df into the normalized tables."""
    conn = psycopg2.connect(DB_URL)
    cursor = conn.cursor()
    inserted = 0
    skipped = 0
    for _, row in df.iterrows():
        # 1. Upsert lookup tables
        category_id = upsert_lookup(cursor, "categories", "name", row["category"])
        # 2. Insert main record with ON CONFLICT DO NOTHING
        cursor.execute(
            """
            INSERT INTO records (id, record_date, value, category_id)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (id) DO NOTHING
            """,
            (row["id"], row["record_date"], row["value"], category_id)
        )
        if cursor.rowcount == 1:
            inserted += 1
        else:
            skipped += 1
    conn.commit()
    cursor.close()
    conn.close()
    print(f"Inserted: {inserted} rows | Skipped (duplicates): {skipped} rows")

# TODO: load CSV, call port_dataframe
print("port_data.py skeleton loaded — implementation pending")

---
## Section 6 — The Streamlit Dashboard

### 6.1 Overview

The dashboard is a `streamlit_app.py` file hosted on **Streamlit Community Cloud**. It auto-deploys whenever code is pushed to the main branch of the GitHub repository. Users open a URL in their browser — nothing to install.

The dashboard connects directly to Supabase PostgreSQL and runs read-only queries. It must never write to the database.

---

### 6.2 Deployment

1. Connect your GitHub repo to [Streamlit Community Cloud](https://streamlit.io/cloud)
2. Set `SUPABASE_URL` and `SUPABASE_KEY` as Streamlit secrets (equivalent to GitHub Secrets — never stored in code)
3. Every push to `main` triggers a redeploy automatically

---

### 6.3 Dashboard Requirements

The dashboard should include:

| Feature | Description |
|---|---|
| Summary metrics | Total records, date range of data, last updated timestamp |
| Time series chart | Records or values over time (line or bar chart) |
| Filter controls | Date range picker, category selector, location selector |
| Data table | Paginated view of filtered records |
| Custom query box | Text input for read-only SQL; results displayed as a table |

---

### 6.4 `streamlit_app.py` — Skeleton

In [ ]:
# streamlit_app.py — skeleton (not yet implemented)

import streamlit as st
import pandas as pd
from supabase import create_client

# Streamlit secrets replace environment variables when deployed
SUPABASE_URL = st.secrets["SUPABASE_URL"]
SUPABASE_KEY = st.secrets["SUPABASE_KEY"]

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

@st.cache_data(ttl=300)  # cache query results for 5 minutes
def query(sql: str) -> pd.DataFrame:
    """Run a read-only SQL query and return results as a DataFrame."""
    # Enforce read-only: reject any statement that isn't SELECT
    if not sql.strip().upper().startswith("SELECT"):
        raise ValueError("Only SELECT statements are permitted.")
    result = supabase.rpc("run_query", {"sql": sql}).execute()
    return pd.DataFrame(result.data)

# --- Layout ---
st.title("Project Data Dashboard")
st.caption("Read-only view of the pipeline database")

# Summary metrics
# TODO: query for total rows, date range, last updated

# Filters
col1, col2 = st.columns(2)
with col1:
    date_range = st.date_input("Date range", [])
with col2:
    category = st.selectbox("Category", ["All"])  # TODO: populate from DB

# Main chart
# TODO: time series chart using st.line_chart or plotly

# Data table
# TODO: paginated filtered results

# Custom query
st.subheader("Custom SQL query (SELECT only)")
user_sql = st.text_area("Enter a SELECT statement", height=100)
if st.button("Run query") and user_sql:
    try:
        result_df = query(user_sql)
        st.dataframe(result_df)
    except ValueError as e:
        st.error(str(e))

print("streamlit_app.py skeleton loaded — implementation pending")

---
## Section 7 — Scheduled Updates via GitHub Actions

### 7.1 Overview

GitHub Actions runs our incremental update pipeline automatically on a schedule. No server to manage — GitHub provides the compute for free (within limits).

The workflow file lives at `.github/workflows/update_pipeline.yml` and is version-controlled like all other code.

---

### 7.2 What the Workflow Does

On each scheduled run, GitHub Actions:

1. Spins up a fresh Ubuntu environment
2. Checks out the latest code from the `main` branch
3. Installs Python dependencies from `requirements.txt`
4. Injects secrets (`SUPABASE_URL`, `SUPABASE_KEY`, `API_KEY`, `DATABASE_URL`) as environment variables
5. Runs `fetch_new.py` — fetches new records, saves to Supabase Storage, exports `TARGET_FILE`
6. Runs `clean_data.py` with `TARGET_FILE` — cleans and saves cleaned CSV
7. Runs `port_data.py` with `TARGET_FILE` — ports cleaned data into the database

If any step fails, subsequent steps do not run and GitHub sends an email notification to the repository owner.

---

### 7.3 Workflow File

In [ ]:
# This cell displays the GitHub Actions workflow YAML for reference.
# The actual file lives at .github/workflows/update_pipeline.yml

workflow_yaml = """
name: Scheduled pipeline update

on:
  schedule:
    - cron: '0 6 * * *'   # runs daily at 06:00 UTC
  workflow_dispatch:       # allows manual trigger from GitHub UI

jobs:
  update:
    runs-on: ubuntu-latest

    steps:
      - name: Check out code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Fetch new data
        env:
          SUPABASE_URL: ${{ secrets.SUPABASE_URL }}
          SUPABASE_KEY: ${{ secrets.SUPABASE_KEY }}
          API_KEY:      ${{ secrets.API_KEY }}
          DATABASE_URL: ${{ secrets.DATABASE_URL }}
        run: python fetch_new.py

      - name: Clean data
        env:
          SUPABASE_URL: ${{ secrets.SUPABASE_URL }}
          SUPABASE_KEY: ${{ secrets.SUPABASE_KEY }}
          TARGET_FILE:  ${{ env.TARGET_FILE }}
        run: python clean_data.py

      - name: Port data into database
        env:
          SUPABASE_URL: ${{ secrets.SUPABASE_URL }}
          SUPABASE_KEY: ${{ secrets.SUPABASE_KEY }}
          DATABASE_URL: ${{ secrets.DATABASE_URL }}
          TARGET_FILE:  ${{ env.TARGET_FILE }}
        run: python port_data.py
"""

print(workflow_yaml)

### 7.4 Secrets Management

All credentials are stored as **GitHub Secrets** (Settings → Secrets and variables → Actions). They are never written into source files or printed in logs.

Required secrets for this project:

| Secret Name | What It Is |
|---|---|
| `SUPABASE_URL` | Your Supabase project URL |
| `SUPABASE_KEY` | Supabase service role key (has full DB access) |
| `DATABASE_URL` | Direct PostgreSQL connection string (from Supabase settings) |
| `API_KEY` | API key for the data source |

---
## Section 8 — Setup Checklist

Use this checklist when setting up the project for the first time:

### 8.1 One-Time Setup

- [ ] Create GitHub repository; invite all team members
- [ ] Create Supabase project; add team members
- [ ] Create two Storage buckets in Supabase: `raw` and `cleaned`
- [ ] Add all four secrets to GitHub repository settings
- [ ] Run `setup_db.py` locally (once, with credentials exported) to create the schema
- [ ] Run `fetch_all.py` to pull full historical data
- [ ] Run `clean_data.py` for each raw CSV produced
- [ ] Run `port_data.py` for each cleaned CSV produced
- [ ] Verify row counts in Supabase table viewer
- [ ] Connect repo to Streamlit Community Cloud; set Streamlit secrets
- [ ] Confirm dashboard loads and displays data
- [ ] Push `.github/workflows/update_pipeline.yml`; trigger manually once to confirm it works

### 8.2 Ongoing (Automated)

- GitHub Actions runs daily at 06:00 UTC
- Dashboard reflects new data within minutes of each successful run
- Monitor the Actions tab in GitHub for any failures